In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, recall_score, precision_score
from imblearn.over_sampling import SMOTE
import joblib
import warnings
warnings.filterwarnings('ignore')

df_flood = pd.read_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\final_flood_dataset_with_municipalities.csv')
df_flood['date'] = pd.to_datetime(df_flood['date'], format='mixed')

feature_cols_flood = [
    'temperature', 'rainfall', 'humidity', 'wind_speed',
    'discharge', 'rainfall_roll3', 'rainfall_roll7',
    'discharge_roll3', 'discharge_roll7',
    'month', 'location_encoded', 'terrain_encoded'
]

train_data = df_flood[df_flood['date'].dt.year <= 2020]
test_data  = df_flood[df_flood['date'].dt.year >= 2021]

X_train = train_data[feature_cols_flood]
y_train = train_data['flood_risk_label']
X_test  = test_data[feature_cols_flood]
y_test  = test_data['flood_risk_label']

print(f'Training: {len(X_train)} rows')
print(f'Testing:  {len(X_test)} rows')
print('\nTraining label counts:')
print(y_train.value_counts())

Training: 1917750 rows
Testing:  456500 rows

Training label counts:
flood_risk_label
0    1907326
1       9619
2        805
Name: count, dtype: int64


In [3]:
sampling_strategy = {
    1: 45000,   
    2: 18000,   
}

smote = SMOTE(random_state=42, k_neighbors=5, sampling_strategy=sampling_strategy)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('Before SMOTE:', y_train.value_counts().to_dict())
print('After SMOTE:', pd.Series(y_train_sm).value_counts().to_dict())

Before SMOTE: {0: 1907326, 1: 9619, 2: 805}
After SMOTE: {0: 1907326, 1: 45000, 2: 18000}


In [4]:
xgb_model_v2 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0,
    n_jobs=-1
)
xgb_model_v2.fit(X_train_sm, y_train_sm)

probs = xgb_model_v2.predict_proba(X_test)
print('Training complete!')
print('Max High probability in test set:', probs[:, 2].max())

Training complete!
Max High probability in test set: 0.8239103


In [5]:
for threshold in [0.01, 0.02, 0.05, 0.10, 0.15]:
    custom_preds = np.where(
        probs[:, 2] > threshold, 2,
        np.argmax(probs[:, :2], axis=1)
    )
    print(f'\n--- Threshold = {threshold} ---')
    print(classification_report(y_test, custom_preds, target_names=['Low','Medium','High'], zero_division=0))


--- Threshold = 0.01 ---
              precision    recall  f1-score   support

         Low       0.99      0.93      0.96    450075
      Medium       0.37      0.02      0.05      6105
        High       0.01      0.85      0.02       320

    accuracy                           0.92    456500
   macro avg       0.46      0.60      0.34    456500
weighted avg       0.98      0.92      0.95    456500


--- Threshold = 0.02 ---
              precision    recall  f1-score   support

         Low       0.99      0.95      0.97    450075
      Medium       0.37      0.02      0.05      6105
        High       0.01      0.74      0.02       320

    accuracy                           0.94    456500
   macro avg       0.46      0.57      0.35    456500
weighted avg       0.98      0.94      0.96    456500


--- Threshold = 0.05 ---
              precision    recall  f1-score   support

         Low       0.99      0.97      0.98    450075
      Medium       0.37      0.02      0.05      61

In [6]:
xgb_predictions_v2 = np.where(probs[:, 2] > 0.01, 2, np.argmax(probs[:, :2], axis=1))

print(classification_report(y_test, xgb_predictions_v2, target_names=['Low','Medium','High']))

importance = pd.Series(xgb_model_v2.feature_importances_, index=feature_cols_flood).sort_values(ascending=False)
print('\nFeature importance:')
print(importance)

              precision    recall  f1-score   support

         Low       0.99      0.93      0.96    450075
      Medium       0.37      0.02      0.05      6105
        High       0.01      0.85      0.02       320

    accuracy                           0.92    456500
   macro avg       0.46      0.60      0.34    456500
weighted avg       0.98      0.92      0.95    456500


Feature importance:
rainfall_roll7      0.223401
terrain_encoded     0.131764
discharge           0.116995
discharge_roll7     0.092979
month               0.071015
location_encoded    0.068958
discharge_roll3     0.065325
rainfall_roll3      0.064762
rainfall            0.061157
temperature         0.043340
humidity            0.038492
wind_speed          0.021813
dtype: float32


In [8]:
from sklearn.preprocessing import LabelEncoder

le_location_v2 = LabelEncoder()
le_location_v2.fit(df_flood['location'])

le_terrain_v2 = LabelEncoder()
le_terrain_v2.fit(df_flood['terrain'])

check = pd.DataFrame({
    'location': df_flood['location'],
    'existing_encoded': df_flood['location_encoded'],
    'rebuilt_encoded': le_location_v2.transform(df_flood['location'])
})
print('Location encoder matches?', (check['existing_encoded'] == check['rebuilt_encoded']).all())

check2 = pd.DataFrame({
    'terrain': df_flood['terrain'],
    'existing_encoded': df_flood['terrain_encoded'],
    'rebuilt_encoded': le_terrain_v2.transform(df_flood['terrain'])
})
print('Terrain encoder matches?', (check2['existing_encoded'] == check2['rebuilt_encoded']).all())

Location encoder matches? True
Terrain encoder matches? True


In [9]:
HIGH_RISK_THRESHOLD_V2 = 0.01

joblib.dump(xgb_model_v2, r'C:\Nepal_Flood_Project\Data\Districts_77\flood_xgboost_model_with_municipalities.pkl')
joblib.dump(HIGH_RISK_THRESHOLD_V2, r'C:\Nepal_Flood_Project\Data\Districts_77\flood_xgboost_threshold_with_municipalities.pkl')
joblib.dump(le_location_v2, r'C:\Nepal_Flood_Project\Data\Districts_77\location_encoder_with_municipalities.pkl')
joblib.dump(le_terrain_v2, r'C:\Nepal_Flood_Project\Data\Districts_77\terrain_encoder_with_municipalities.pkl')

print('Flood model with municipalities saved!')

Flood model with municipalities saved!


In [10]:
df_landslide = pd.read_csv(r'C:\Nepal_Flood_Project\Data\Districts_77\final_landslide_dataset_with_municipalities.csv')
df_landslide['date'] = pd.to_datetime(df_landslide['date'], format='mixed')

feature_cols_landslide = [
    'temperature', 'rainfall', 'humidity', 'wind_speed',
    'rain_3day', 'rain_7day', 'soil_moisture', 'slope',
    'month', 'terrain_encoded', 'location_encoded'
]

train_data_ls = df_landslide[df_landslide['date'].dt.year <= 2020]
test_data_ls  = df_landslide[df_landslide['date'].dt.year >= 2021]

X_train_ls = train_data_ls[feature_cols_landslide]
y_train_ls = train_data_ls['landslide_risk_label']
X_test_ls  = test_data_ls[feature_cols_landslide]
y_test_ls  = test_data_ls['landslide_risk_label']

print(f'Training: {len(X_train_ls)} rows')
print(f'Testing:  {len(X_test_ls)} rows')
print('\nTraining label counts:')
print(y_train_ls.value_counts())

Training: 1917750 rows
Testing:  456500 rows

Training label counts:
landslide_risk_label
0    1895656
1      21291
2        803
Name: count, dtype: int64


In [11]:
sampling_strategy_ls = {
    1: 90000,   
    2: 24000,   
}

smote_ls = SMOTE(random_state=42, k_neighbors=5, sampling_strategy=sampling_strategy_ls)
X_train_sm_ls, y_train_sm_ls = smote_ls.fit_resample(X_train_ls, y_train_ls)

print('Before SMOTE:', y_train_ls.value_counts().to_dict())
print('After SMOTE:', pd.Series(y_train_sm_ls).value_counts().to_dict())

Before SMOTE: {0: 1895656, 1: 21291, 2: 803}
After SMOTE: {0: 1895656, 1: 90000, 2: 24000}


In [12]:
xgb_model_ls_v2 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0,
    n_jobs=-1
)
xgb_model_ls_v2.fit(X_train_sm_ls, y_train_sm_ls)

probs_ls = xgb_model_ls_v2.predict_proba(X_test_ls)
print('Training complete!')
print('Max High probability in test set:', probs_ls[:, 2].max())

Training complete!
Max High probability in test set: 0.8157206


In [13]:
for threshold in [0.01, 0.02, 0.05, 0.10, 0.15]:
    custom_preds = np.where(
        probs_ls[:, 2] > threshold, 2,
        np.argmax(probs_ls[:, :2], axis=1)
    )
    print(f'\n--- Threshold = {threshold} ---')
    print(classification_report(y_test_ls, custom_preds, target_names=['Low','Medium','High'], zero_division=0))


--- Threshold = 0.01 ---
              precision    recall  f1-score   support

         Low       0.97      0.92      0.94    432620
      Medium       0.46      0.10      0.16     22741
        High       0.03      0.89      0.05      1139

    accuracy                           0.88    456500
   macro avg       0.48      0.63      0.38    456500
weighted avg       0.94      0.88      0.90    456500


--- Threshold = 0.02 ---
              precision    recall  f1-score   support

         Low       0.97      0.94      0.95    432620
      Medium       0.46      0.10      0.16     22741
        High       0.03      0.82      0.06      1139

    accuracy                           0.90    456500
   macro avg       0.49      0.62      0.39    456500
weighted avg       0.94      0.90      0.91    456500


--- Threshold = 0.05 ---
              precision    recall  f1-score   support

         Low       0.96      0.96      0.96    432620
      Medium       0.46      0.10      0.16     227

In [14]:
xgb_predictions_ls_v2 = np.where(probs_ls[:, 2] > 0.01, 2, np.argmax(probs_ls[:, :2], axis=1))

print(classification_report(y_test_ls, xgb_predictions_ls_v2, target_names=['Low','Medium','High']))

importance_ls = pd.Series(xgb_model_ls_v2.feature_importances_, index=feature_cols_landslide).sort_values(ascending=False)
print('\nFeature importance:')
print(importance_ls)

              precision    recall  f1-score   support

         Low       0.97      0.92      0.94    432620
      Medium       0.46      0.10      0.16     22741
        High       0.03      0.89      0.05      1139

    accuracy                           0.88    456500
   macro avg       0.48      0.63      0.38    456500
weighted avg       0.94      0.88      0.90    456500


Feature importance:
terrain_encoded     0.367727
humidity            0.181189
slope               0.093224
month               0.083975
rainfall            0.066089
rain_7day           0.061314
soil_moisture       0.058084
location_encoded    0.045282
temperature         0.020808
rain_3day           0.013226
wind_speed          0.009081
dtype: float32


In [15]:
HIGH_RISK_THRESHOLD_LS_V2 = 0.01

joblib.dump(xgb_model_ls_v2, r'C:\Nepal_Flood_Project\Data\Districts_77\landslide_xgboost_model_with_municipalities.pkl')
joblib.dump(HIGH_RISK_THRESHOLD_LS_V2, r'C:\Nepal_Flood_Project\Data\Districts_77\landslide_xgboost_threshold_with_municipalities.pkl')

le_location_ls_v2 = LabelEncoder()
le_location_ls_v2.fit(df_landslide['location'])
le_terrain_ls_v2 = LabelEncoder()
le_terrain_ls_v2.fit(df_landslide['terrain'])

joblib.dump(le_location_ls_v2, r'C:\Nepal_Flood_Project\Data\Districts_77\landslide_location_encoder_with_municipalities.pkl')
joblib.dump(le_terrain_ls_v2, r'C:\Nepal_Flood_Project\Data\Districts_77\landslide_terrain_encoder_with_municipalities.pkl')

print('Landslide model with municipalities saved!')

Landslide model with municipalities saved!
